# LeetCode 1933 — Check if String Is Decomposable Into Value-Equal Substrings bằng Python

**Link đề:** https://leetcode.com/problems/check-if-string-is-decomposable-into-value-equal-substrings/
**Độ khó:** Easy · **Chủ đề:** String

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
class Solution:
    def isDecomposable(self, s: str) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu khái niệm **value-equal substring** (chuỗi con mà mọi ký tự bằng nhau) và **run** (đoạn liên tiếp cùng ký tự).
2. Biết tách một chuỗi thành các run bằng vòng lặp hai con trỏ và bằng `itertools.groupby`.
3. Đi từ cách trực giác (thử cắt mảnh dài 2 hoặc 3 bằng đệ quy + memo) tới insight **chỉ cần nhìn `độ dài run % 3`**.
4. Trace từng bước, chứng minh tính đúng, phân tích Time/Space.
5. Tự giải các biến thể: đếm run, run-length encoding, tổng quát hoá độ dài mảnh.

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — cắt chuỗi số thành các mảnh "toàn một chữ số"

Một chuỗi gọi là **value-equal** nếu mọi ký tự trong nó giống nhau. Ví dụ `"1111"`, `"33"` là value-equal; `"123"` thì không.

Cho chuỗi chữ số `s`. Ta muốn cắt `s` thành các **chuỗi con liên tiếp** (không bỏ ký tự, không đổi thứ tự) sao cho:
- mỗi mảnh là value-equal;
- **đúng một** mảnh có độ dài `2`;
- **tất cả** các mảnh còn lại có độ dài `3`.

Trả về `True` nếu cắt được, ngược lại `False`.

| `s` | Output | Giải thích |
|---|:---:|---|
| `"000111000"` | `False` | chỉ cắt được `"000" "111" "000"` — không có mảnh dài 2 |
| `"00011111222"` | `True` | `"000" "111" "11" "222"` |
| `"011100022233"` | `False` | ký tự `"0"` đầu tiên đứng một mình, không ghép được thành mảnh dài 2 hay 3 |

**Constraints (theo hiểu biết về bài):** `1 <= s.length <= 1000`, `s` chỉ gồm chữ số `'0'..'9'`.

Câu hỏi đặt ra trước khi code:
- Hai mảnh liền nhau có được dùng **cùng một chữ số** không? (Có — ví dụ 2 cắt `"11111"` thành `"111" + "11"`.)
- Có cần thật sự "thử cắt" không, hay chỉ cần đếm?

In [1]:
# Kiểm tra bằng mắt: ghép các mảnh ở ví dụ 2 lại có ra đúng s không?
pieces = ["000", "111", "11", "222"]
s = "00011111222"
print("".join(pieces) == s)                 # True: không mất ký tự, không đổi thứ tự
print([len(p) for p in pieces])             # [3, 3, 2, 3]: đúng một mảnh dài 2
print(all(len(set(p)) == 1 for p in pieces))  # True: mỗi mảnh chỉ có một loại ký tự

True
[3, 3, 2, 3]
True


## 2. Nền tảng Python cần dùng

### 2.1. Chuỗi, chỉ số (index) và so sánh ký tự

Chuỗi `str` giống một dãy ký tự đánh số từ `0`. `s[i]` đọc ký tự thứ `i`; `len(s)` là số ký tự.
So sánh `s[i] == s[i + 1]` cho biết hai ký tự liền nhau có giống nhau không — đây là phép kiểm tra cốt lõi của bài.

In [2]:
s = "00011"
print(len(s))          # 5
print(s[0], s[2], s[3])  # 0 0 1
for i in range(len(s) - 1):          # dừng ở len(s)-2 để s[i+1] không vượt biên
    print(i, s[i], s[i + 1], "giống" if s[i] == s[i + 1] else "KHÁC")

5
0 0 1
0 0 0 giống
1 0 0 giống
2 0 1 KHÁC
3 1 1 giống


### 2.2. Slicing `s[a:b]` và `set`

- `s[a:b]` lấy đoạn từ chỉ số `a` tới **trước** `b` (độ dài `b - a`).
- `set(x)` tạo tập hợp các phần tử **khác nhau**. Một chuỗi là value-equal khi `len(set(chuoi)) == 1`.

In [3]:
s = "00011111222"
print(s[0:3], s[3:6], s[6:8], s[8:11])   # 000 111 11 222
print(set("111"), len(set("111")))        # {'1'} 1  -> value-equal
print(set("112"), len(set("112")))        # 2 phần tử -> không value-equal

000 111 11 222
{'1'} 1
{'1', '2'} 2


### 2.3. Khái niệm **run** — đoạn liên tiếp dài nhất cùng ký tự

Viết `s = "00011111222"` thành các khối: `000 | 11111 | 222`. Mỗi khối gọi là một **run** (còn gọi là "nhóm liên tiếp", "maximal block").
Run là khái niệm then chốt: một mảnh value-equal **không bao giờ vắt qua ranh giới** giữa hai run, vì hai bên ranh giới là hai chữ số khác nhau.

Cách tách run bằng **hai con trỏ**: `i` đứng ở đầu run, `j` chạy sang phải khi `s[j] == s[i]`. Khi dừng, run là `s[i:j]`, độ dài `j - i`; rồi gán `i = j`.

In [4]:
def runs_two_pointers(s):
    result = []
    i = 0
    while i < len(s):
        j = i
        while j < len(s) and s[j] == s[i]:   # kiểm tra j < len(s) TRƯỚC khi đọc s[j]
            j += 1
        result.append((s[i], j - i))          # (ký tự, độ dài run)
        i = j
    return result

print(runs_two_pointers("00011111222"))   # [('0', 3), ('1', 5), ('2', 3)]
print(runs_two_pointers("011100022233"))  # [('0', 1), ('1', 3), ('0', 3), ('2', 3), ('3', 2)]

[('0', 3), ('1', 5), ('2', 3)]
[('0', 1), ('1', 3), ('0', 3), ('2', 3), ('3', 2)]


### 2.4. `itertools.groupby` — thư viện chuẩn tách run sẵn

`groupby(s)` duyệt chuỗi và gom các phần tử **liền nhau** bằng nhau thành một nhóm. Mỗi lần lặp trả về cặp `(key, group)`:
- `key` là ký tự của nhóm;
- `group` là một iterator (bộ duyệt) — muốn biết độ dài phải đếm, ví dụ `len(list(group))` hoặc `sum(1 for _ in group)`.

Lưu ý: `groupby` chỉ gom phần tử **liền nhau**, nên `"0110"` cho ba nhóm `0, 11, 0` chứ không gom hai số 0 lại — đúng thứ ta cần.

In [5]:
from itertools import groupby

for key, group in groupby("0110"):
    items = list(group)
    print(key, items, len(items))
print([(k, len(list(g))) for k, g in groupby("00011111222")])

0 ['0'] 1
1 ['1', '1'] 2
0 ['0'] 1
[('0', 3), ('1', 5), ('2', 3)]


### 2.5. Phép chia lấy dư `%` và chia nguyên `//`

- `a // 3` là số lần 3 "vừa" trong `a`; `a % 3` là phần dư (luôn là `0`, `1` hoặc `2` với `a >= 0`).
- Ví dụ `5 = 3*1 + 2` nên `5 // 3 = 1`, `5 % 3 = 2`.

Ta sẽ thấy phần dư theo 3 quyết định toàn bộ bài toán.

In [6]:
for L in range(1, 10):
    print(f"L={L}: L//3={L // 3}, L%3={L % 3}")

L=1: L//3=0, L%3=1
L=2: L//3=0, L%3=2
L=3: L//3=1, L%3=0
L=4: L//3=1, L%3=1
L=5: L//3=1, L%3=2
L=6: L//3=2, L%3=0
L=7: L//3=2, L%3=1
L=8: L//3=2, L%3=2
L=9: L//3=3, L%3=0


## 3. Tính tay — một run dài `L` có cắt được không?

Vì mảnh không vắt qua ranh giới run, ta xét **từng run độc lập**. Một run dài `L` phải được cắt thành các mảnh dài 3 và (có thể) mảnh dài 2:

| `L` | Cách cắt | Số mảnh dài 2 cần dùng |
|---:|---|:---:|
| 1 | không cắt được | — (thất bại) |
| 2 | `2` | 1 |
| 3 | `3` | 0 |
| 4 | `2+2` (hai mảnh 2 — vi phạm "đúng một") | — (thất bại) |
| 5 | `3+2` | 1 |
| 6 | `3+3` | 0 |
| 7 | `3+2+2` hoặc `3+3+1`… đều không hợp lệ | — (thất bại) |
| 8 | `3+3+2` | 1 |

Quy luật hiện ra: chỉ cần nhìn `L % 3`.
- `L % 3 == 0` → dùng toàn mảnh 3, **không tốn** mảnh 2.
- `L % 3 == 2` → bắt buộc tốn **đúng một** mảnh 2.
- `L % 3 == 1` → cần ít nhất hai mảnh 2 (vì `1 ≡ 2+2 (mod 3)`), không bao giờ hợp lệ.

Nhưng đừng vội tin — trước tiên viết một cách "thử cắt" trực giác để làm **oracle** (lời giải đối chiếu), rồi mới kiểm chứng quy luật.

In [7]:
# Kiểm chứng bằng vét cạn: L = 3a + 2b với a, b >= 0; cần b nhỏ nhất là bao nhiêu?
for L in range(1, 13):
    options = [b2 for b2 in range(0, L // 2 + 1) if (L - 2 * b2) >= 0 and (L - 2 * b2) % 3 == 0]
    print(L, "L%3 =", L % 3, "-> số mảnh 2 có thể:", options)

1 L%3 = 1 -> số mảnh 2 có thể: []
2 L%3 = 2 -> số mảnh 2 có thể: [1]
3 L%3 = 0 -> số mảnh 2 có thể: [0]
4 L%3 = 1 -> số mảnh 2 có thể: [2]
5 L%3 = 2 -> số mảnh 2 có thể: [1]
6 L%3 = 0 -> số mảnh 2 có thể: [0, 3]
7 L%3 = 1 -> số mảnh 2 có thể: [2]
8 L%3 = 2 -> số mảnh 2 có thể: [1, 4]
9 L%3 = 0 -> số mảnh 2 có thể: [0, 3]
10 L%3 = 1 -> số mảnh 2 có thể: [2, 5]
11 L%3 = 2 -> số mảnh 2 có thể: [1, 4]
12 L%3 = 0 -> số mảnh 2 có thể: [0, 3, 6]


Bảng trên cho thấy: với `L % 3 == 1`, số mảnh 2 nhỏ nhất là `2` (ví dụ `L=4 → b=2`, `L=7 → b=2`). Vì cả chuỗi chỉ được phép có **đúng một** mảnh 2, run như vậy làm hỏng ngay.

## 4. Cách 1 — Trực giác: thử cắt từ trái sang phải (đệ quy)

Người mới thường nghĩ: "đứng ở vị trí `i`, ta có hai lựa chọn: cắt một mảnh dài 3 hoặc một mảnh dài 2 (nếu chưa dùng mảnh 2)". Thử cả hai, cái nào đi tới cuối chuỗi là thành công.

Hàm `can(i, used_two)` nghĩa là: *phần còn lại `s[i:]` có cắt được không, biết rằng ta đã dùng mảnh 2 hay chưa*.
- Nếu `i == len(s)`: hết chuỗi → thành công **chỉ khi** `used_two` là `True` (phải có đúng một mảnh 2).
- Thử mảnh 3: cần `s[i] == s[i+1] == s[i+2]`.
- Thử mảnh 2: cần chưa dùng và `s[i] == s[i+1]`.

In [8]:
def is_decomposable_recursive(s):
    n = len(s)

    def can(i, used_two):
        if i == n:
            return used_two
        # thử mảnh dài 3
        if i + 3 <= n and s[i] == s[i + 1] == s[i + 2]:
            if can(i + 3, used_two):
                return True
        # thử mảnh dài 2 (chỉ khi chưa dùng)
        if not used_two and i + 2 <= n and s[i] == s[i + 1]:
            if can(i + 2, True):
                return True
        return False

    return can(0, False)

print(is_decomposable_recursive("000111000"))     # False
print(is_decomposable_recursive("00011111222"))   # True
print(is_decomposable_recursive("011100022233"))  # False

False
True
False


### 4.1. Chuỗi so sánh `a == b == c` trong Python

`s[i] == s[i+1] == s[i+2]` được Python hiểu là `s[i] == s[i+1] and s[i+1] == s[i+2]`. Đây là **chained comparison** (so sánh nối chuỗi), rất tiện để kiểm tra 3 ký tự giống nhau.

In [9]:
print("1" == "1" == "1")   # True
print("1" == "1" == "2")   # False
print(1 < 2 < 3, 1 < 3 < 2)  # True False

True
False
True False


### 4.2. Nút thắt của cách đệ quy

Mỗi vị trí có thể rẽ 2 nhánh, nên số lời gọi có thể bùng nổ. Tuy nhiên trạng thái chỉ là cặp `(i, used_two)`: có `n + 1` giá trị `i` và 2 giá trị `used_two` → tối đa `2(n+1)` trạng thái khác nhau. Nếu **ghi nhớ** (memoization) kết quả từng trạng thái, mỗi trạng thái chỉ tính một lần.

Python có sẵn `functools.lru_cache` (hoặc `cache`) để ghi nhớ tự động: đặt `@lru_cache(None)` trên hàm, lần gọi thứ hai với cùng đối số sẽ trả ngay kết quả đã lưu.

Ngoài ra đệ quy sâu tới `n/2` tầng (≈500 với `n = 1000`) — vẫn dưới giới hạn mặc định 1000 của Python nhưng khá sát, là một rủi ro nữa.

In [10]:
from functools import lru_cache

calls = 0

@lru_cache(None)
def slow_square(x):
    global calls
    calls += 1
    return x * x

print(slow_square(7), slow_square(7), slow_square(8))
print("số lần thực sự tính:", calls)   # 2, vì lần gọi slow_square(7) thứ hai lấy từ cache

49 49 64
số lần thực sự tính: 2


In [11]:
def is_decomposable_memo(s):
    n = len(s)

    @lru_cache(None)
    def can(i, used_two):
        if i == n:
            return used_two
        if i + 3 <= n and s[i] == s[i + 1] == s[i + 2] and can(i + 3, used_two):
            return True
        if not used_two and i + 2 <= n and s[i] == s[i + 1] and can(i + 2, True):
            return True
        return False

    return can(0, False)

for t in ["000111000", "00011111222", "011100022233", "22", "2", "11111"]:
    print(repr(t), is_decomposable_memo(t))

'000111000' False
'00011111222' True
'011100022233' False
'22' True
'2' False
'11111' True


Cách memo đã là `O(n)` thời gian, nhưng code dài, dùng đệ quy và bộ nhớ `O(n)` cho cache. Ta muốn một lời giải **một lượt duyệt, O(1) bộ nhớ, không đệ quy**.

## 5. Bước đột phá — chỉ cần đếm số run có `L % 3 == 2`

Từ mục 3 ta có hai sự thật:
1. Mảnh không vắt qua ranh giới run → mỗi run được cắt **độc lập**.
2. Một run dài `L` cần `0` mảnh 2 nếu `L % 3 == 0`, đúng `1` mảnh 2 nếu `L % 3 == 2`, và **không hợp lệ** nếu `L % 3 == 1`.

Vậy thuật toán:
- Duyệt từng run; nếu có run với `L % 3 == 1` → `False` ngay.
- Đếm `twos` = số run có `L % 3 == 2`.
- Trả về `twos == 1`.

In [12]:
# Áp quy tắc bằng tay trên ví dụ 2
for ch, L in runs_two_pointers("00011111222"):
    print(ch, L, "L%3 =", L % 3)
twos = sum(1 for _, L in runs_two_pointers("00011111222") if L % 3 == 2)
print("twos =", twos, "->", twos == 1)

0 3 L%3 = 0
1 5 L%3 = 2
2 3 L%3 = 0
twos = 1 -> True


## 6. Lời giải chính — đếm run một lượt, mổ xẻ từng đoạn

Ta không cần lưu danh sách run; chỉ cần biến `run_len` đếm độ dài run hiện tại.

**Đoạn 1 — khởi tạo:** `twos = 0`, `run_len = 0`.

**Đoạn 2 — duyệt từng vị trí `i`:** tăng `run_len`. Run **kết thúc tại `i`** khi `i` là ký tự cuối, hoặc `s[i] != s[i+1]`.

**Đoạn 3 — khi run kết thúc:** xét `run_len % 3`:
- `== 1` → trả `False`;
- `== 2` → `twos += 1`, nếu `twos > 1` có thể trả `False` sớm;
- reset `run_len = 0`.

**Đoạn 4 — kết thúc:** trả `twos == 1`.

In [13]:
# Đoạn 2: cách nhận biết "run kết thúc tại i"
s = "0011122"
for i in range(len(s)):
    end_here = (i == len(s) - 1) or (s[i] != s[i + 1])
    print(i, s[i], "kết thúc run" if end_here else "")

0 0 
1 0 kết thúc run
2 1 
3 1 
4 1 kết thúc run
5 2 
6 2 kết thúc run


Chú ý thứ tự trong `(i == len(s) - 1) or (s[i] != s[i + 1])`: `or` **đánh giá ngắn mạch** (short-circuit) — nếu vế trái đã `True` thì Python không tính vế phải, nhờ đó không đọc `s[i+1]` khi `i` là vị trí cuối (tránh `IndexError`).

In [14]:
def is_decomposable(s):
    twos = 0
    run_len = 0
    n = len(s)
    for i in range(n):
        run_len += 1
        if i == n - 1 or s[i] != s[i + 1]:   # run kết thúc tại i
            r = run_len % 3
            if r == 1:
                return False
            if r == 2:
                twos += 1
                if twos > 1:
                    return False
            run_len = 0
    return twos == 1

for t in ["000111000", "00011111222", "011100022233"]:
    print(repr(t), is_decomposable(t))

'000111000' False
'00011111222' True
'011100022233' False


## 7. TRACE từng bước

Ví dụ `s = "00011111222"` (n = 11). Cột "kết thúc?" là kết quả điều kiện `i == n-1 or s[i] != s[i+1]`.

| i | s[i] | run_len sau `+= 1` | kết thúc? | run_len % 3 | twos | Hành động |
|---:|:---:|:---:|:---:|:---:|:---:|---|
| 0 | 0 | 1 | không | | 0 | |
| 1 | 0 | 2 | không | | 0 | |
| 2 | 0 | 3 | có (s[3]='1') | 0 | 0 | run 3 → không tốn mảnh 2, reset |
| 3 | 1 | 1 | không | | 0 | |
| 4–6 | 1 | 2..4 | không | | 0 | |
| 7 | 1 | 5 | có (s[8]='2') | 2 | 1 | run 5 = 3+2 → twos = 1 |
| 8–9 | 2 | 1..2 | không | | 1 | |
| 10 | 2 | 3 | có (cuối chuỗi) | 0 | 1 | run 3 |
| kết thúc | | | | | 1 | trả `twos == 1` → `True` |

Hàm `trace_decomposable` dưới đây in đúng các trạng thái đó; ví dụ thứ hai `"0011100"` có hai run dài 2 → dừng sớm vì `twos > 1`.

In [15]:
def trace_decomposable(s):
    print(f"s = {s!r}")
    twos = 0
    run_len = 0
    n = len(s)
    for i in range(n):
        run_len += 1
        end_here = i == n - 1 or s[i] != s[i + 1]
        if not end_here:
            print(f"  i={i:>2} s[i]={s[i]} run_len={run_len}")
            continue
        r = run_len % 3
        print(f"  i={i:>2} s[i]={s[i]} run_len={run_len} -> RUN KẾT THÚC, run_len%3={r}", end="")
        if r == 1:
            print(" -> dư 1, không cắt được -> False")
            return False
        if r == 2:
            twos += 1
            print(f" -> cần một mảnh 2, twos={twos}", end="")
            if twos > 1:
                print(" -> quá một mảnh 2 -> False")
                return False
        print()
        run_len = 0
    print(f"  Hết chuỗi: twos={twos} -> {twos == 1}")
    return twos == 1

trace_decomposable("00011111222")
print()
trace_decomposable("0011100")

s = '00011111222'
  i= 0 s[i]=0 run_len=1
  i= 1 s[i]=0 run_len=2
  i= 2 s[i]=0 run_len=3 -> RUN KẾT THÚC, run_len%3=0
  i= 3 s[i]=1 run_len=1
  i= 4 s[i]=1 run_len=2
  i= 5 s[i]=1 run_len=3
  i= 6 s[i]=1 run_len=4
  i= 7 s[i]=1 run_len=5 -> RUN KẾT THÚC, run_len%3=2 -> cần một mảnh 2, twos=1
  i= 8 s[i]=2 run_len=1
  i= 9 s[i]=2 run_len=2
  i=10 s[i]=2 run_len=3 -> RUN KẾT THÚC, run_len%3=0
  Hết chuỗi: twos=1 -> True

s = '0011100'
  i= 0 s[i]=0 run_len=1
  i= 1 s[i]=0 run_len=2 -> RUN KẾT THÚC, run_len%3=2 -> cần một mảnh 2, twos=1
  i= 2 s[i]=1 run_len=1
  i= 3 s[i]=1 run_len=2
  i= 4 s[i]=1 run_len=3 -> RUN KẾT THÚC, run_len%3=0
  i= 5 s[i]=0 run_len=1
  i= 6 s[i]=0 run_len=2 -> RUN KẾT THÚC, run_len%3=2 -> cần một mảnh 2, twos=2 -> quá một mảnh 2 -> False


False

### 7.1. Trace một nhánh đặc biệt: run dài 4

`s = "11110"`: run `1111` có `L = 4`, `4 % 3 == 1`. Dù "nhìn qua" có vẻ cắt được `"11" + "11"`, đó là **hai** mảnh 2 — vi phạm. Còn `"111" + "1"` thì mảnh `"1"` dài 1, không hợp lệ. Thuật toán dừng ngay ở run đầu tiên.

In [16]:
trace_decomposable("11110")
print("Oracle đệ quy đồng ý:", is_decomposable_recursive("11110"))

s = '11110'
  i= 0 s[i]=1 run_len=1
  i= 1 s[i]=1 run_len=2
  i= 2 s[i]=1 run_len=3
  i= 3 s[i]=1 run_len=4 -> RUN KẾT THÚC, run_len%3=1 -> dư 1, không cắt được -> False
Oracle đệ quy đồng ý: False


## 8. Các cách viết khác

### 8.1. Cách 3 — `itertools.groupby` (ngắn gọn nhất)

Ý tưởng giống hệt mục 6 nhưng để `groupby` tách run giúp ta.

In [17]:
def is_decomposable_groupby(s):
    twos = 0
    for _, group in groupby(s):
        L = sum(1 for _ in group)   # đếm độ dài nhóm mà không cần tạo list
        if L % 3 == 1:
            return False
        if L % 3 == 2:
            twos += 1
    return twos == 1

for t in ["000111000", "00011111222", "011100022233", "22", "2"]:
    print(repr(t), is_decomposable_groupby(t))

'000111000' False
'00011111222' True
'011100022233' False
'22' True
'2' False


### 8.2. Cách 4 — Biểu thức chính quy (regex) tìm run

Module `re` (thư viện chuẩn) có thể tìm run bằng mẫu `(\d)\1*`:
- `(\d)` bắt một chữ số vào **nhóm 1**;
- `\1*` nghĩa là "lặp lại đúng ký tự của nhóm 1, 0 lần trở lên".

`re.finditer` trả các match; `m.group(0)` là cả run. Cách này dùng để luyện regex, không nhanh hơn cách 6.

In [18]:
import re

print([m.group(0) for m in re.finditer(r"(\d)\1*", "00011111222")])

def is_decomposable_regex(s):
    lengths = [len(m.group(0)) for m in re.finditer(r"(\d)\1*", s)]
    if any(L % 3 == 1 for L in lengths):
        return False
    return sum(1 for L in lengths if L % 3 == 2) == 1

print(is_decomposable_regex("00011111222"), is_decomposable_regex("000111000"))

['000', '11111', '222']
True False


## 9. Code nộp LeetCode (Python 3)

Bản chính: một lượt duyệt, đếm `run_len`, không đệ quy.

In [19]:
class Solution:
    def isDecomposable(self, s: str) -> bool:
        twos = 0
        run_len = 0
        n = len(s)
        for i in range(n):
            run_len += 1
            if i == n - 1 or s[i] != s[i + 1]:
                r = run_len % 3
                if r == 1:
                    return False
                if r == 2:
                    twos += 1
                    if twos > 1:
                        return False
                run_len = 0
        return twos == 1


sol = Solution()
assert sol.isDecomposable("000111000") is False
assert sol.isDecomposable("00011111222") is True
assert sol.isDecomposable("011100022233") is False
print("3 ví dụ của đề PASS")

3 ví dụ của đề PASS


Bản ngắn với `groupby` (cũng nộp được):

In [20]:
from itertools import groupby

class SolutionGroupby:
    def isDecomposable(self, s: str) -> bool:
        twos = 0
        for _, g in groupby(s):
            r = sum(1 for _ in g) % 3
            if r == 1:
                return False
            twos += r == 2          # True được cộng như 1, False như 0
        return twos == 1

print(SolutionGroupby().isDecomposable("00011111222"))
print(True + True, False + 0)   # minh hoạ: bool là số nguyên con

True
2 0


## 10. Vì sao thuật toán đúng?

**Bổ đề 1 (không vắt qua ranh giới):** mỗi mảnh value-equal nằm trọn trong một run. Lý do: nếu một mảnh chứa cả `s[j]` và `s[j+1]` với `s[j] != s[j+1]` thì mảnh có hai ký tự khác nhau — mâu thuẫn. Vậy một cách cắt toàn chuỗi chính là ghép các cách cắt của từng run.

**Bổ đề 2 (một run dài `L`):** nếu run được cắt thành `a` mảnh 3 và `b` mảnh 2 thì `L = 3a + 2b`, suy ra `L ≡ 2b (mod 3)`.
- `b = 0` → `L ≡ 0`; `b = 1` → `L ≡ 2`; `b >= 2` thì vi phạm đề.
- Do đó run có `L % 3 == 1` **không thể** cắt hợp lệ; run có `L % 3 == 0` bắt buộc `b = 0`; run có `L % 3 == 2` bắt buộc `b = 1`. Ngược lại mọi trường hợp đó đều thực hiện được (cắt `L // 3` mảnh 3, thêm một mảnh 2 nếu dư 2).

**Kết luận:** tổng số mảnh 2 toàn chuỗi **bị ép buộc** bằng số run có dư 2 — không có lựa chọn nào khác. Đề cần tổng đó bằng đúng 1 và không có run dư 1. Đó chính là điều thuật toán kiểm tra.

**Invariant của vòng lặp:** trước mỗi bước `i`, `twos` = số run **đã kết thúc** có dư 2, `run_len` = số ký tự của run đang dở ở cuối `s[:i]`, và chưa run nào đã kết thúc có dư 1.

## 11. Time / Space Complexity

Gọi `n = len(s)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Đệ quy thử cắt (không memo) | tệ nhất có thể lớn hơn tuyến tính | `O(n)` stack | dễ nghĩ, nhưng rủi ro |
| 2. Đệ quy + `lru_cache` | `O(n)` (≤ `2(n+1)` trạng thái) | `O(n)` cache + stack | đúng, dài dòng |
| 3. Đếm `run_len` một lượt (chính) | `O(n)` | `O(1)` | nên nộp |
| 4. `groupby` | `O(n)` | `O(1)` | ngắn nhất |
| 5. Regex | `O(n)` | `O(số run)` cho list độ dài | luyện regex |

Mỗi ký tự được tăng `run_len` đúng một lần và so sánh với ký tự kế tiếp đúng một lần → `O(n)`.

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu nhiều cách

In [21]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.isDecomposable("000111000"), False)
check("đề 2", sol.isDecomposable("00011111222"), True)
check("đề 3", sol.isDecomposable("011100022233"), False)
check("một ký tự", sol.isDecomposable("7"), False)
check("đúng một mảnh 2", sol.isDecomposable("77"), True)
check("chỉ mảnh 3", sol.isDecomposable("777"), False)
check("run 4 (dư 1)", sol.isDecomposable("7777"), False)
check("run 5 = 3+2", sol.isDecomposable("77777"), True)
check("hai run dư 2", sol.isDecomposable("1122"), False)
check("run dư 1 ở cuối", sol.isDecomposable("11122" + "3"), False)
check("cùng chữ số nhưng tách run", sol.isDecomposable("1110111"), False)

PASS đề 1: False
PASS đề 2: True
PASS đề 3: False
PASS một ký tự: False
PASS đúng một mảnh 2: True
PASS chỉ mảnh 3: False
PASS run 4 (dư 1): False
PASS run 5 = 3+2: True
PASS hai run dư 2: False
PASS run dư 1 ở cuối: False
PASS cùng chữ số nhưng tách run: False


### 12.1. Kiểm thử đối chiếu toàn bộ chuỗi nhỏ

Dùng `itertools.product` sinh **mọi** chuỗi trên bảng chữ `"01"` độ dài 1..10 (tổng 2046 chuỗi) và so sánh 4 cách với oracle đệ quy. Nếu quy luật `% 3` sai ở đâu, assert sẽ bắt được.

In [22]:
from itertools import product

count = 0
for length in range(1, 11):
    for tup in product("01", repeat=length):
        t = "".join(tup)
        expected = is_decomposable_recursive(t)
        assert is_decomposable_memo(t) == expected, t
        assert is_decomposable(t) == expected, t
        assert is_decomposable_groupby(t) == expected, t
        assert is_decomposable_regex(t) == expected, t
        assert Solution().isDecomposable(t) == expected, t
        count += 1
print("Đã đối chiếu", count, "chuỗi — tất cả khớp")

Đã đối chiếu 2046 chuỗi — tất cả khớp


In [23]:
# Kiểm thử ngẫu nhiên có seed với chuỗi dài hơn, nhiều chữ số
import random
random.seed(1933)
for _ in range(300):
    parts = []
    for _ in range(random.randint(1, 8)):
        parts.append(random.choice("0123456789") * random.randint(1, 8))
    t = "".join(parts)
    assert is_decomposable(t) == is_decomposable_memo(t), t
print("300 test ngẫu nhiên PASS")

300 test ngẫu nhiên PASS


## 13. Sai lầm người mới hay mắc

### Lỗi 1 — Quên rằng "đúng một" mảnh 2 (trả `twos >= 1` hoặc `twos <= 1`)

```python
return twos <= 1   # SAI: "777" có twos = 0 vẫn trả True
```
Đề yêu cầu **đúng một** mảnh dài 2 → phải là `twos == 1`.

### Lỗi 2 — Đọc `s[i+1]` khi `i` là vị trí cuối

```python
if s[i] != s[i + 1] or i == n - 1:   # SAI thứ tự: tính s[i+1] trước -> IndexError
```
Sửa: đặt `i == n - 1` **trước** trong biểu thức `or`.

### Lỗi 3 — Quên xử lý run cuối cùng

Nhiều bạn chỉ "chốt" run khi gặp ký tự khác, nên run cuối (không có ký tự khác phía sau) bị bỏ sót. Cell dưới minh hoạ.

In [24]:
def buggy_forget_last_run(s):
    twos, run_len = 0, 1
    for i in range(1, len(s)):
        if s[i] == s[i - 1]:
            run_len += 1
        else:                       # chỉ chốt run khi gặp ký tự KHÁC
            if run_len % 3 == 1:
                return False
            twos += run_len % 3 == 2
            run_len = 1
    return twos == 1                # QUÊN chốt run cuối!

print("buggy '1100' :", buggy_forget_last_run("1100"), "| đúng:", is_decomposable("1100"))
print("buggy '00011':", buggy_forget_last_run("00011"), "| đúng:", is_decomposable("00011"))

buggy '1100' : True | đúng: False
buggy '00011': False | đúng: True


Bản lỗi trả `True` cho `"1100"` (đúng phải là `False` vì có hai run dư 2) và `False` cho `"00011"` (đúng phải là `True`), vì run cuối `"00"` / `"11"` chưa từng được xét. Sửa: sau vòng lặp, chốt thêm run cuối — hoặc dùng điều kiện "kết thúc tại `i`" như lời giải chính.

### Lỗi 4 — Tưởng có thể "mượn" ký tự từ run bên cạnh

Ví dụ `"0111"`: có bạn nghĩ cắt `"01"` + `"11"`. Nhưng `"01"` không value-equal. Mảnh **không bao giờ** vắt qua ranh giới run (Bổ đề 1).

### Lỗi 5 — Gộp các run cùng chữ số ở xa nhau

Dùng `Counter(s)` đếm tổng số lần xuất hiện mỗi chữ số là **sai**: `"10011"` có ba chữ `1` (chia hết cho 3) và hai chữ `0` (dư 2) nên `Counter` kết luận `True`; nhưng thực tế các run là `1 | 00 | 11` — run `1` đầu dài 1 không cắt được. Phải đếm theo run liên tiếp.

In [25]:
from collections import Counter

def buggy_counter(s):
    twos = 0
    for c, total in Counter(s).items():
        if total % 3 == 1:
            return False
        twos += total % 3 == 2
    return twos == 1

for t in ["10011", "1110111"]:
    print(repr(t), "buggy:", buggy_counter(t), "| đúng:", is_decomposable(t))

'10011' buggy: True | đúng: False
'1110111' buggy: False | đúng: False


`"10011"`: bản lỗi trả `True`, đáp đúng là `False`. `"1110111"`: cả hai cùng `False` nhưng chỉ là **tình cờ** (bản lỗi thấy `0:1` dư 1). Một test pass không chứng minh code đúng — bài học: **đếm theo run, không đếm toàn cục**.

## 14. Mẹo nhận dạng pattern

- Đề nói "**substring/chuỗi con liên tiếp** mà mọi ký tự giống nhau" → nghĩ ngay tới **tách run** (hai con trỏ hoặc `groupby`).
- Đề hỏi "cắt thành mảnh độ dài cố định / tổng các độ dài" → thử viết `L = 3a + 2b` và xét **phần dư** (`%`). Phần dư thường biến một bài "thử mọi cách cắt" thành bài **đếm**.
- Khi chưa chắc quy luật, viết **oracle vét cạn** (đệ quy) và **đối chiếu tự động** trên mọi input nhỏ — đây là kỹ năng kiểm thử rất đáng giá.
- Bài cùng họ: 1869 (Longer Contiguous Segments of Ones than Zeros), 1446 (Consecutive Characters), 443 (String Compression), 1957 (Delete Characters to Make Fancy String).

## 15. Biến thể

### 15.1. Trả về chính cách cắt (danh sách mảnh) hoặc `[]` nếu không được

Vì cách cắt từng run bị ép buộc, ta xây được ngay: mỗi run cắt thành `L // 3` mảnh 3, thêm mảnh 2 nếu dư 2.

In [26]:
def decompose(s):
    pieces = []
    twos = 0
    for ch, g in groupby(s):
        L = sum(1 for _ in g)
        if L % 3 == 1:
            return []
        pieces.extend([ch * 3] * (L // 3))
        if L % 3 == 2:
            pieces.append(ch * 2)
            twos += 1
    return pieces if twos == 1 else []

print(decompose("00011111222"))   # ['000', '111', '11', '222']
print(decompose("000111000"))     # []

['000', '111', '11', '222']
[]


### 15.2. Tổng quát: mảnh dài `k`, cho phép đúng `m` mảnh dài `k - 1`

Lập luận tương tự với phần dư theo `k`: một run dài `L` cần `b` mảnh `(k-1)` với `L = k·a + (k-1)·b`, tức `L ≡ -b (mod k)`. Với `b < k`, `b = (-L) % k`. Ta cộng `b` của mọi run và so với `m` (giả định `k ≥ 2`, `m < k` để mỗi run có cách cắt duy nhất).

In [27]:
def decomposable_general(s, k, m):
    total_b = 0
    for _, g in groupby(s):
        L = sum(1 for _ in g)
        b_needed = (-L) % k              # số mảnh (k-1) nhỏ nhất làm L chia hết
        if (k - 1) * b_needed > L:       # không đủ ký tự để tạo chừng ấy mảnh
            return False
        total_b += b_needed
    return total_b == m

print(decomposable_general("00011111222", 3, 1))   # True — trùng bài gốc
print(decomposable_general("0000" + "111", 4, 1))  # True: 0000 | 111
print(decomposable_general("11", 4, 1))           # False: cần mảnh 3 mà chỉ có 2 ký tự

True
True
False


## 16. Bài tập (TODO — tự làm, không chạy tự động)

**Bài 1 (dự đoán output):** không chạy code, hãy cho biết `[(k, len(list(g))) for k, g in groupby("aabbbac")]` in ra gì.

**Bài 2 (tính tay):** với `s = "55566666667777"`, hãy liệt kê run, `L % 3` và kết luận `True/False`.

**Bài 3 (điền chỗ trống):** hoàn thiện `exercise_run_lengths(s)` trả list độ dài các run bằng hai con trỏ.

In [28]:
def exercise_run_lengths(s):
    result = []
    i = 0
    while i < len(s):
        # TODO 1: cho j chạy từ i sang phải khi s[j] == s[i]
        # TODO 2: thêm độ dài run vào result
        # TODO 3: nhảy i tới j
        pass
    return result

# Tự làm xong rồi mới mở test:
# assert exercise_run_lengths("00011111222") == [3, 5, 3]
# assert exercise_run_lengths("7") == [1]

**Bài 4 (tự viết lại):** viết `exercise_is_decomposable(s)` **không** dùng `groupby`, chỉ một vòng `for` và biến `run_len`.

In [29]:
def exercise_is_decomposable(s):
    # TODO: viết lại lời giải chính
    pass

# assert exercise_is_decomposable("00011111222") is True
# assert exercise_is_decomposable("000111000") is False
# assert exercise_is_decomposable("7777") is False

**Bài 5 (biến thể):** "Mọi mảnh đều dài đúng 3" — viết `exercise_all_threes(s)` trả `True` khi `s` cắt được thành các mảnh value-equal **toàn** dài 3.

**Bài 6 (mini challenge):** viết `exercise_rle(s)` nén chuỗi theo run-length encoding: `"aaabcc"` → `"a3b1c2"`.

In [30]:
def exercise_all_threes(s):
    # TODO
    pass

def exercise_rle(s):
    # TODO
    pass

# assert exercise_all_threes("000111") is True
# assert exercise_all_threes("00011") is False
# assert exercise_rle("aaabcc") == "a3b1c2"
# assert exercise_rle("z") == "z1"

## 17. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | `groupby` chỉ gom phần tử **liền nhau** | chữ `a` cuối là nhóm riêng | 4 nhóm: `aa`, `bbb`, `a`, `c` |
| 2 | tách run trước | run `555`, `6666666`, `7777` | dư lần lượt 0, 1, 1 |
| 3 | `j = i` rồi `while` | điều kiện `j < len(s) and s[j] == s[i]` | `result.append(j - i); i = j` |
| 4 | run kết thúc khi nào? | `i == n-1 or s[i] != s[i+1]` | xét `run_len % 3`, reset `run_len = 0` |
| 5 | mọi run phải chia hết cho 3 | `all(...)` với `groupby` | `all(sum(1 for _ in g) % 3 == 0 for _, g in groupby(s))` |
| 6 | tách run, ghép `ký tự + str(độ dài)` | `"".join(...)` | `"".join(k + str(len(list(g))) for k, g in groupby(s))` |

## 18. Đáp án tham khảo — chỉ xem sau khi tự làm

**Bài 1:** `[('a', 2), ('b', 3), ('a', 1), ('c', 1)]`.

**Bài 2:** run `555` (L=3, dư 0), `6666666` (L=7, dư 1) → `False` ngay, không cần xét `7777`.

In [31]:
print([(k, len(list(g))) for k, g in groupby("aabbbac")])   # đáp án bài 1
print(is_decomposable("55566666667777"))                     # đáp án bài 2: False


def answer_run_lengths(s):
    result = []
    i = 0
    while i < len(s):
        j = i
        while j < len(s) and s[j] == s[i]:
            j += 1
        result.append(j - i)
        i = j
    return result


def answer_is_decomposable(s):
    twos, run_len, n = 0, 0, len(s)
    for i in range(n):
        run_len += 1
        if i == n - 1 or s[i] != s[i + 1]:
            if run_len % 3 == 1:
                return False
            if run_len % 3 == 2:
                twos += 1
            run_len = 0
    return twos == 1


def answer_all_threes(s):
    return all(sum(1 for _ in g) % 3 == 0 for _, g in groupby(s))


def answer_rle(s):
    return "".join(k + str(len(list(g))) for k, g in groupby(s))


assert answer_run_lengths("00011111222") == [3, 5, 3]
assert answer_run_lengths("7") == [1]
assert answer_is_decomposable("00011111222") is True
assert answer_is_decomposable("000111000") is False
assert answer_is_decomposable("7777") is False
assert answer_all_threes("000111") is True
assert answer_all_threes("00011") is False
assert answer_rle("aaabcc") == "a3b1c2"
assert answer_rle("z") == "z1"
for length in range(1, 9):
    for tup in product("01", repeat=length):
        t = "".join(tup)
        assert answer_is_decomposable(t) == is_decomposable_recursive(t)
print("Đáp án tham khảo PASS toàn bộ test!")

[('a', 2), ('b', 3), ('a', 1), ('c', 1)]
False
Đáp án tham khảo PASS toàn bộ test!


## 19. Template nhớ nhanh + checklist submit

```python
class Solution:
    def isDecomposable(self, s: str) -> bool:
        twos = run_len = 0
        for i in range(len(s)):
            run_len += 1
            if i == len(s) - 1 or s[i] != s[i + 1]:   # chốt run
                if run_len % 3 == 1: return False
                twos += run_len % 3 == 2
                run_len = 0
        return twos == 1
```

**Câu thần chú:** *tách run → nhìn `L % 3` → dư 1 là chết, dư 2 tốn một mảnh 2 → cần đúng một.*

**Checklist trước khi submit:**
- [ ] Đã xét **run cuối** chưa?
- [ ] Không đọc `s[i+1]` khi `i = n-1` (đặt `i == n-1` trước trong `or`)?
- [ ] Trả `twos == 1`, không phải `<= 1` hay `>= 1`?
- [ ] Đếm theo **run liên tiếp**, không dùng `Counter` toàn chuỗi?
- [ ] Test `"77"` (True), `"777"` (False), `"7777"` (False), `"77777"` (True)?